# Download and Process MACA Data

This notebook will download MACAv2 (or v3, if access becomes possible!), check for NaNs, and engineer the same features as the PRISM data.

## Step 1: Set up code space

In [1]:
# import libraries

# File management
import glob
import os
import pathlib
from pathlib import Path
import sys # for importing src files

# Downloading
from tqdm.notebook import tqdm # progress bar
import requests # may need for api access

# Data Management
import numpy as np
import pandas as pd
import xarray as xr

# Geospatial data
import geopandas as gpd
import rasterio
import rioxarray as rxr

# Plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas

print("Imports Complete!")

Imports Complete!


In [2]:
# Force Jupyter to reload external modules automatically before executing a cell
# I used Gemini to help me set up the connection to my src files

# force reimporting of src scripts
%load_ext autoreload
%autoreload 2

# Find the parent directory of this notebook (the repo root) and add it to Python's search path
repo_root = str(Path.cwd().parent)
if repo_root not in sys.path:
    sys.path.append(repo_root)

# Now Python can see the src folder
# import just specific functions
from src import maca, features

print('src import complete!')

src import complete!


In [3]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')
os.makedirs(proj_dir, exist_ok=True)

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)

cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')
os.makedirs(raw_data_dir, exist_ok=True)

In [4]:
# Import study boundary
mhw_train_path = Path(cleaned_data_dir, 'boundaries', 'mhw_buff_clip_gdf.parquet')
mhw_gdf = gpd.read_parquet(mhw_train_path)

In [5]:
mhw_gdf.total_bounds

array([-113.95841725,   44.34404648, -110.46445972,   46.46949588])

## Step 2: Download MACA data

Missouri Headwaters watershed bounds:
-113.95841725,   44.34404648, -110.46445972,   46.46949588

*Model Ensemble*
- RCP 4.5 (8.5 is no longer realistic)
- Max temp delta JJA, Precip delta DJF

Cool/Wet: CNRM-CM5 (+0.905in, +4.451f)
Cool/Dry: MRI-CGCM3 (+0.155in, +2.279f) - alternate IPSL-CM5B-LR
Warm/Wet: CSIRO-Mk3-6-0 (+0.777in, +7.713f)
Warm/Dry: CanESM2 (+0.361in, +7.528f)

- MAM temp and precip delta
Cool/Wet: CNRM-CM5 (+1.027in, +2.650f)
Cool/Dry: GFDL-ESM2M (+0.086in, +1.302f)
Warm/Wet: MIROC-ESM (+2.234in, +9.141f) - alternate CanESM2 (+1.511in, +5.945f) GO WITH CanESM2
Warm/Dry: BNU-ESM (+0.238in, +7.076f) - go with IPSL-CM5A-MR (+0.147in, +5.537f)

In [6]:
# data dir for MACA data

maca_dir = os.path.join(raw_data_dir, 'maca_data')
os.makedirs(maca_dir, exist_ok = True)

# pattern
maca_pattern = os.path.join(maca_dir, '*.nc')

Need to list and explain model selection here.

In [7]:
# Set desired MACA parameters

models = {'CNRM-CM5'}#, 'GFDL-ESM2M', 'CanESM2', 'IPSL-CM5A-MR'}
variables = {'tasmax'}#, 'tasmin', 'pr'}

In [24]:
# Download MACA data

test = maca.download_maca_data(
    site_gdf=mhw_gdf, site_name='missouri_headwaters',
    models=models, rcp_value='rcp45', variables=variables, start_date= '2074-12-01', end_date='2075-01-01', maca_dir=maca_dir
)
#     mhw_gdf, 'mhw',
#     models, 'rcp45', variables,
#     '2074-10-01', '2075-06-01',
#     maca_dir
# )

Processing: CNRM-CM5 (tasmax): 100%|██████████| 1/1 [00:01<00:00,  1.83s/it]

tasmax | CNRM-CM5 | 2074-12-01 - 2075-01-01 | rcp45 has been downloaded!.


In [ ]:
# check downloaded file
ds_test = xr.open_dataset(test[0])
print(ds_test)
print("\nCoordinates:", list(ds_test.coords))
print("Data variables:", list(ds_test.data_vars))
print("Min:", float(ds_test[list(ds_test.data_vars)[1]].min()))
print("Max:", float(ds_test[list(ds_test.data_vars)[1]].max()))

<xarray.Dataset> Size: 543kB
Dimensions:      (time: 32, lat: 51, lon: 83)
Coordinates:
  * time         (time) datetime64[ns] 256B 2074-12-01 2074-12-02 ... 2075-01-01
  * lat          (lat) float64 408B 44.35 44.4 44.44 44.48 ... 46.35 46.4 46.44
  * lon          (lon) float64 664B -113.9 -113.9 -113.8 ... -110.5 -110.5
    spatial_ref  int64 8B ...
Data variables:
    crs          int64 8B ...
    tasmax       (time, lat, lon) float32 542kB ...
Attributes: (12/46)
    description:                     Multivariate Adaptive Constructed Analog...
    id:                              MACAv2-METDATA
    naming_authority:                edu.uidaho.reacch
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    Metadata_Link:                   
    cdm_data_type:                   GRID
    ...                              ...
    contributor_role:                Postdoctoral Fellow
    publisher_name:                  
    publisher_email:                 
    publisher_url:

In [8]:
# set up params for full download

models = {'CNRM-CM5', 'GFDL-ESM2M', 'CanESM2', 'IPSL-CM5A-MR'}
variables = {'tasmax', 'tasmin', 'pr'}

In [9]:
# complete download
maca_paths = maca.download_maca_data(
    # set site gdf and name
    site_gdf=mhw_gdf, site_name='missouri_headwaters',
    # set models, rcp value, and variables of interest
    models=models, rcp_value='rcp45', variables=variables, 
    # set start date and end date
    start_date= '2045-10-01', end_date='2065-01-01', 
    # set dir
    maca_dir=maca_dir
)

Processing: IPSL-CM5A-MR (tasmax): 100%|██████████| 12/12 [00:00<00:00, 420.71it/s]

tasmin | CNRM-CM5 | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
pr | CNRM-CM5 | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
tasmax | CNRM-CM5 | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
tasmin | GFDL-ESM2M | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
pr | GFDL-ESM2M | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
tasmax | GFDL-ESM2M | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
tasmin | CanESM2 | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
pr | CanESM2 | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
tasmax | CanESM2 | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
tasmin | IPSL-CM5A-MR | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
pr | IPSL-CM5A-MR | 2045-10-01 - 2065-01-01 | rcp45 | already exists.
tasmax | IPSL-CM5A-MR | 2045-10-01 - 2065-01-01 | rcp45 | already exists.


In [10]:
# check the paths
maca_paths

['C:\\Users\\Raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\data\\raw\\maca_data\\maca_tasmin_CNRM-CM5_missouri_headwaters_rcp45_2045-10-01_2065-01-01_CONUS_daily.nc',
 'C:\\Users\\Raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\data\\raw\\maca_data\\maca_pr_CNRM-CM5_missouri_headwaters_rcp45_2045-10-01_2065-01-01_CONUS_daily.nc',
 'C:\\Users\\Raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\data\\raw\\maca_data\\maca_tasmax_CNRM-CM5_missouri_headwaters_rcp45_2045-10-01_2065-01-01_CONUS_daily.nc',
 'C:\\Users\\Raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\data\\raw\\maca_data\\maca_tasmin_GFDL-ESM2M_missouri_headwaters_rcp45_2045-10-01_2065-01-01_CONUS_daily.nc',
 'C:\\Users\\Raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\data\\raw\\maca_data\\maca_pr_GFDL-ESM2M_missouri_headwaters_rcp45_2045-10-01_2065-01-01_C

## Step 3: Feature Engineering and Preparation for ML Application

In [ ]:
# load models into datasets and perform feature engineering
# I'm using a dict here to make this code more DRY and flexible - it relies on the 'models' variable above

# initialize dict
datasets = {}

# loop through model names and load data into lazy datasets
for model in models:
    # Print status update
    print(f"Loading and processing {model}...")

    # make search pattern and get paths
    pattern = os.path.join(maca_dir, f'maca_*_{model}_*.nc')
    file_paths = glob.glob(pattern)

    # open model ds and combine tasmin, tasmax, and pr
    model_ds = xr.open_mfdataset(
        # open the file paths
        file_paths, 
        # combine by coords (time)
        combine = 'by_coords',
        # decode coords so that crs is read as a coord
        decode_coords = 'all')

    # standardize variable names
    rename_map = {k: v for k, v in {'pr': 'ppt', 'tasmax': 'tmax', 'tasmin': 'tmin'}.items() if k in model_ds}
    if rename_map:
        model_ds = model_ds.rename(rename_map)

    # perform feature engineering
    model_ds = features.engineer_all_features(model_ds)

    # append to dict
    datasets[model] = model_ds

Loading and processing CNRM-CM5...


C:\Users\Raini\AppData\Local\Temp\ipykernel_20200\3453718816.py:17: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  model_ds = xr.open_mfdataset(
C:\Users\Raini\AppData\Local\Temp\ipykernel_20200\3453718816.py:17: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  model_ds = xr.open_mfdataset(


Loading and processing GFDL-ESM2M...


C:\Users\Raini\AppData\Local\Temp\ipykernel_20200\3453718816.py:17: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  model_ds = xr.open_mfdataset(


Loading and processing CanESM2...


C:\Users\Raini\AppData\Local\Temp\ipykernel_20200\3453718816.py:17: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  model_ds = xr.open_mfdataset(


Loading and processing IPSL-CM5A-MR...


C:\Users\Raini\AppData\Local\Temp\ipykernel_20200\3453718816.py:17: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  model_ds = xr.open_mfdataset(


In [13]:
# check out a model ds
datasets['GFDL-ESM2M']

<xarray.Dataset> Size: 962MB
Dimensions:                   (time: 4734, lat: 51, lon: 83)
Coordinates:
  * time                      (time) datetime64[ns] 38kB 2045-10-01 ... 2065-...
    water_year                (time) int64 38kB 2046 2046 2046 ... 2065 2065
    dowy                      (time) int64 38kB 1 2 3 4 5 6 ... 89 90 91 92 93
  * lat                       (lat) float64 408B 44.35 44.4 44.44 ... 46.4 46.44
  * lon                       (lon) float64 664B -113.9 -113.9 ... -110.5 -110.5
    spatial_ref               int64 8B 0
Data variables:
    ppt                       (time, lat, lon) float32 80MB dask.array<chunksize=(4734, 51, 83), meta=np.ndarray>
    tmax                      (time, lat, lon) float32 80MB dask.array<chunksize=(4734, 51, 83), meta=np.ndarray>
    tmin                      (time, lat, lon) float32 80MB dask.array<chunksize=(4734, 51, 83), meta=np.ndarray>
    tmean_c                   (time, lat, lon) float32 80MB dask.array<chunksize=(4734, 51, 83), meta=np.ndarray>
    wy_cumul_ppt_mm           (time, lat, lon) float32 80MB dask.array<chunksize=(244, 51, 83), meta=np.ndarray>
    tmin_roll3_c              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 26, 42), meta=np.ndarray>
    tmax_roll3_c              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 26, 42), meta=np.ndarray>
    ppt_roll3_mm              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 26, 42), meta=np.ndarray>
    tmin_roll7_c              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 17, 28), meta=np.ndarray>
    tmax_roll7_c              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 17, 28), meta=np.ndarray>
    ppt_roll7_mm              (time, lat, lon) float32 80MB dask.array<chunksize=(244, 17, 28), meta=np.ndarray>
    wy_cumul_pos_degree_days  (time, lat, lon) float32 80MB dask.array<chunksize=(244, 51, 83), meta=np.ndarray>
Attributes: (12/46)
    description:                     Multivariate Adaptive Constructed Analog...
    id:                              MACAv2-METDATA
    naming_authority:                edu.uidaho.reacch
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    Metadata_Link:                   
    cdm_data_type:                   GRID
    ...                              ...
    contributor_role:                Postdoctoral Fellow
    publisher_name:                  
    publisher_email:                 
    publisher_url:                   
    license:                         Creative Commons CC0 1.0 Universal Dedic...
    coordinate_system:               WGS84,EPSG:4326

In [14]:
# import DEM datasets

# set topo dir
topo_rsmp_dir = Path(cleaned_data_dir, 'topo')

# paths
elev_4km_path = Path(topo_rsmp_dir, 'mhw_elevation_4km.tif')
slope_4km_path = Path(topo_rsmp_dir, 'mhw_slope_4km.tif')
aspect_4km_path = Path(topo_rsmp_dir, 'mhw_aspect_4km.tif')
aspect_north_4km_path = Path(topo_rsmp_dir, 'mhw_northness_4km.tif')
aspect_east_4km_path = Path(topo_rsmp_dir, 'mhw_eastness_4km.tif')

# import
mhw_elev_4km_da = rxr.open_rasterio(elev_4km_path)
mhw_slope_4km_da = rxr.open_rasterio(slope_4km_path)
mhw_aspect_4km_da = rxr.open_rasterio(aspect_4km_path)
mhw_aspect_4km_north_da = rxr.open_rasterio(aspect_north_4km_path)
mhw_aspect_4km_east_da = rxr.open_rasterio(aspect_east_4km_path)

# merge into one dataset
dem_ds = xr.Dataset({
    'elevation': mhw_elev_4km_da,
    'slope': mhw_slope_4km_da,
    'aspect': mhw_aspect_4km_da,
    'aspect_north': mhw_aspect_4km_north_da,
    'aspect_east': mhw_aspect_4km_east_da
})

# make sure dataset is in ESPG: 4326
if dem_ds.rio.crs.to_epsg() != 4326:
    dem_ds = dem_ds.rio.reproject("EPSG:4326")

In [15]:
# check the ds
dem_ds

<xarray.Dataset> Size: 77kB
Dimensions:       (band: 1, x: 83, y: 51)
Coordinates:
  * band          (band) int64 8B 1
  * x             (x) float64 664B -113.9 -113.9 -113.8 ... -110.6 -110.5 -110.5
  * y             (y) float64 408B 46.46 46.42 46.38 46.33 ... 44.46 44.42 44.38
    spatial_ref   int64 8B 0
Data variables:
    elevation     (band, y, x) int16 8kB ...
    slope         (band, y, x) float32 17kB ...
    aspect        (band, y, x) float32 17kB ...
    aspect_north  (band, y, x) float32 17kB ...
    aspect_east   (band, y, x) float32 17kB ...

In [25]:
# check MACA and DEM lat/lon alignment

# lon
print('DEM lon', dem_ds.x.values[0:5])
print('MACA lon:', datasets['GFDL-ESM2M'].lon.values[0:5])

# lat
print('DEM lat', dem_ds.y.values[0:5])
print('MACA lat:', datasets['GFDL-ESM2M'].lat.values[0:5])

DEM lon [-113.91666873 -113.87500212 -113.8333355  -113.79166889 -113.75000227]
MACA lon: [-113.89733887 -113.85568237 -113.81399536 -113.77233887 -113.73068237]
DEM lat [46.45833687 46.41667026 46.37500364 46.33333703 46.29167041]
MACA lat: [44.35443497 44.39610291 44.43776703 44.47943497 44.52109909]


Need to flip DEM or MACA lat, and match the grids up. Will probably use reproject match there.

In [20]:
# align DEM and MACA grid

# drop band and mask elevation NaNs
dem_clean_ds = dem_ds.squeeze('band', drop = True)
dem_clean_ds['elevation'] = dem_clean_ds['elevation'].where(dem_clean_ds['elevation'] > -1000)

# set maca template
maca_template = datasets['GFDL-ESM2M']['tmin'].isel(time=0)

# merge DEM and MACA grids
dem_align_ds = dem_clean_ds.rio.reproject_match(
    # match to MACA grid
    maca_template,
    # resample w/ bilinear method
    resampling = rasterio.enums.Resampling.bilinear)

# rename
dem_rnm_ds = dem_align_ds.rename({'x': 'lon', 'y': 'lat'})

In [24]:
# now check MACA and DEM lat/lon alignment again

# lon
print('DEM lon', dem_rnm_ds.lon.values[0:5])
print('MACA lon:', datasets['GFDL-ESM2M'].lon.values[0:5])

# lat
print('DEM lat', dem_rnm_ds.lat.values[0:5])
print('MACA lat:', datasets['GFDL-ESM2M'].lat.values[0:5])

DEM lon [-113.89733887 -113.85568237 -113.81399536 -113.77233887 -113.73068237]
MACA lon: [-113.89733887 -113.85568237 -113.81399536 -113.77233887 -113.73068237]
DEM lat [44.35443497 44.39610291 44.43776703 44.47943497 44.52109909]
MACA lat: [44.35443497 44.39610291 44.43776703 44.47943497 44.52109909]


In [ ]:
# double check alignment
print("Lon matches:", np.allclose(dem_rnm_ds.lon, datasets["GFDL-ESM2M"].lon))
print("Lat matches:", np.allclose(dem_rnm_ds.lat, datasets["GFDL-ESM2M"].lat))

Lon matches: True
Lat matches: True


In [26]:
# drop spatial ref and NaNs; flatten dem ds
dem_df = dem_rnm_ds.drop_vars('spatial_ref').to_dataframe().dropna(subset=['elevation'])

In [41]:
# convert MACA ds to df

# get length of time steps in ds
time_steps = datasets['GFDL-ESM2M'].time.values

# set chunks of timesteps to process
chunk_size = 500  # Process 500 days at a time

# initialize df dict
maca_dfs = {}

# loop through models
for model in models:
    print(f"Starting {model} Dataset to DataFrame conversion across {len(time_steps)} time steps...")

    # initialize chunk list
    maca_dem_chunks = []

    # Ensure consistent dimension order across all GCMs
    model_ds = datasets[model].transpose("time", "lat", "lon")

    # loop through chunks and convert to df
    for i in range(0, len(time_steps), chunk_size):
        # Grab the specific block of dates
        time_chunk = time_steps[i : i + chunk_size]
        print(f"Processing chunk {i} to {i + len(time_chunk)}.")

        # Slice the climate dataset for just this time window
        maca_chunk = datasets[model].sel(time=time_chunk)

        # Convert just this small chunk to a dataframe, ensuring dimension order stays constant
        climate_chunk_df = maca_chunk.to_dataframe(dim_order=["time", "lat", "lon"])

        # Join the static terrain features onto this time chunk
        # Pandas automatically matches the 'lat' and 'lon' indices
        maca_dem_chunk = climate_chunk_df.join(dem_df, how="left").dropna()

        # append
        maca_dem_chunks.append(maca_dem_chunk)

    # concatenate the chunks
    maca_dfs[model] = pd.concat(maca_dem_chunks)

    # update and verify dfs are same size
    print(f"Finished {model}: {len(maca_dfs[model]):,} rows.")

# Compete message
print('DataSet flattening complete!')

Starting CNRM-CM5 Dataset to DataFrame conversion across 4734 time steps...
Processing chunk 0 to 500.
Processing chunk 500 to 1000.
Processing chunk 1000 to 1500.
Processing chunk 1500 to 2000.
Processing chunk 2000 to 2500.
Processing chunk 2500 to 3000.
Processing chunk 3000 to 3500.
Processing chunk 3500 to 4000.
Processing chunk 4000 to 4500.
Processing chunk 4500 to 4734.
Finished CNRM-CM5: 10,821,924 rows.
Starting GFDL-ESM2M Dataset to DataFrame conversion across 4734 time steps...
Processing chunk 0 to 500.
Processing chunk 500 to 1000.
Processing chunk 1000 to 1500.
Processing chunk 1500 to 2000.
Processing chunk 2000 to 2500.
Processing chunk 2500 to 3000.
Processing chunk 3000 to 3500.
Processing chunk 3500 to 4000.
Processing chunk 4000 to 4500.
Processing chunk 4500 to 4734.
Finished GFDL-ESM2M: 10,821,924 rows.
Starting CanESM2 Dataset to DataFrame conversion across 4734 time steps...
Processing chunk 0 to 500.
Processing chunk 500 to 1000.
Processing chunk 1000 to 1500.

In [42]:
# check a df
print(maca_dfs['GFDL-ESM2M'].columns)
maca_dfs['GFDL-ESM2M'].head()

# need to drop spatial_ref, crs

Index(['ppt', 'spatial_ref', 'tmax', 'tmin', 'tmean_c', 'water_year', 'dowy',
       'wy_cumul_ppt_mm', 'tmin_roll3_c', 'tmax_roll3_c', 'ppt_roll3_mm',
       'tmin_roll7_c', 'tmax_roll7_c', 'ppt_roll7_mm',
       'wy_cumul_pos_degree_days', 'elevation', 'slope', 'aspect',
       'aspect_north', 'aspect_east'],
      dtype='str')


ppt  spatial_ref       tmax      tmin  \
time       lat       lon                                                  
2045-10-01 44.437767 -112.897354  0.0            0  21.276672  0.786499   
                     -110.897385  0.0            0  18.957458 -1.710175   
                     -110.855713  0.0            0  20.251587 -2.481018   
                     -110.814056  0.0            0  20.666504 -2.614594   
                     -110.772385  0.0            0  19.401642 -2.049164   

                                    tmean_c  water_year  dowy  \
time       lat       lon                                        
2045-10-01 44.437767 -112.897354  11.031586        2046     1   
                     -110.897385   8.623642        2046     1   
                     -110.855713   8.885284        2046     1   
                     -110.814056   9.025955        2046     1   
                     -110.772385   8.676239        2046     1   

                                  wy_cumul_ppt_mm  tmin_roll3_c  tmax_roll3_c  \
time       lat       lon                                                        
2045-10-01 44.437767 -112.897354              0.0      0.786499     21.276672   
                     -110.897385              0.0     -1.710175     18.957458   
                     -110.855713              0.0     -2.481018     20.251587   
                     -110.814056              0.0     -2.614594     20.666504   
                     -110.772385              0.0     -2.049164     19.401642   

                                  ppt_roll3_mm  tmin_roll7_c  tmax_roll7_c  \
time       lat       lon                                                     
2045-10-01 44.437767 -112.897354           0.0      0.786499     21.276672   
                     -110.897385           0.0     -1.710175     18.957458   
                     -110.855713           0.0     -2.481018     20.251587   
                     -110.814056           0.0     -2.614594     20.666504   
                     -110.772385           0.0     -2.049164     19.401642   

                                  ppt_roll7_mm  wy_cumul_pos_degree_days  \
time       lat       lon                                                   
2045-10-01 44.437767 -112.897354           0.0                 11.031586   
                     -110.897385           0.0                  8.623642   
                     -110.855713           0.0                  8.885284   
                     -110.814056           0.0                  9.025955   
                     -110.772385           0.0                  8.676239   

                                    elevation      slope      aspect  \
time       lat       lon                                               
2045-10-01 44.437767 -112.897354  2612.297363  15.732635   87.257339   
                     -110.897385  2529.273438   3.932833  187.365433   
                     -110.855713  2438.088379   5.410627  255.512466   
                     -110.814056  2392.905762   5.706942  160.218735   
                     -110.772385  2478.738525   6.219458  179.198456   

                                  aspect_north  aspect_east  
time       lat       lon                                     
2045-10-01 44.437767 -112.897354     -0.065043     0.795005  
                     -110.897385      0.105646     0.419466  
                     -110.855713     -0.306499     0.789064  
                     -110.814056      0.269574     0.158651  
                     -110.772385      0.019351    -0.721063

In [43]:
# drop unneeded cols

# set cols
drop_cols = ['spatial_ref', 'crs']

# loop through dfs
for model in models:
    maca_dfs[model] = maca_dfs[model].drop(columns = drop_cols, errors = 'ignore')

In [44]:
# check cols again
print(maca_dfs['GFDL-ESM2M'].columns)
maca_dfs['GFDL-ESM2M'].head()

Index(['ppt', 'tmax', 'tmin', 'tmean_c', 'water_year', 'dowy',
       'wy_cumul_ppt_mm', 'tmin_roll3_c', 'tmax_roll3_c', 'ppt_roll3_mm',
       'tmin_roll7_c', 'tmax_roll7_c', 'ppt_roll7_mm',
       'wy_cumul_pos_degree_days', 'elevation', 'slope', 'aspect',
       'aspect_north', 'aspect_east'],
      dtype='str')


ppt       tmax      tmin    tmean_c  \
time       lat       lon                                                
2045-10-01 44.437767 -112.897354  0.0  21.276672  0.786499  11.031586   
                     -110.897385  0.0  18.957458 -1.710175   8.623642   
                     -110.855713  0.0  20.251587 -2.481018   8.885284   
                     -110.814056  0.0  20.666504 -2.614594   9.025955   
                     -110.772385  0.0  19.401642 -2.049164   8.676239   

                                  water_year  dowy  wy_cumul_ppt_mm  \
time       lat       lon                                              
2045-10-01 44.437767 -112.897354        2046     1              0.0   
                     -110.897385        2046     1              0.0   
                     -110.855713        2046     1              0.0   
                     -110.814056        2046     1              0.0   
                     -110.772385        2046     1              0.0   

                                  tmin_roll3_c  tmax_roll3_c  ppt_roll3_mm  \
time       lat       lon                                                     
2045-10-01 44.437767 -112.897354      0.786499     21.276672           0.0   
                     -110.897385     -1.710175     18.957458           0.0   
                     -110.855713     -2.481018     20.251587           0.0   
                     -110.814056     -2.614594     20.666504           0.0   
                     -110.772385     -2.049164     19.401642           0.0   

                                  tmin_roll7_c  tmax_roll7_c  ppt_roll7_mm  \
time       lat       lon                                                     
2045-10-01 44.437767 -112.897354      0.786499     21.276672           0.0   
                     -110.897385     -1.710175     18.957458           0.0   
                     -110.855713     -2.481018     20.251587           0.0   
                     -110.814056     -2.614594     20.666504           0.0   
                     -110.772385     -2.049164     19.401642           0.0   

                                  wy_cumul_pos_degree_days    elevation  \
time       lat       lon                                                  
2045-10-01 44.437767 -112.897354                 11.031586  2612.297363   
                     -110.897385                  8.623642  2529.273438   
                     -110.855713                  8.885284  2438.088379   
                     -110.814056                  9.025955  2392.905762   
                     -110.772385                  8.676239  2478.738525   

                                      slope      aspect  aspect_north  \
time       lat       lon                                                
2045-10-01 44.437767 -112.897354  15.732635   87.257339     -0.065043   
                     -110.897385   3.932833  187.365433      0.105646   
                     -110.855713   5.410627  255.512466     -0.306499   
                     -110.814056   5.706942  160.218735      0.269574   
                     -110.772385   6.219458  179.198456      0.019351   

                                  aspect_east  
time       lat       lon                       
2045-10-01 44.437767 -112.897354     0.795005  
                     -110.897385     0.419466  
                     -110.855713     0.789064  
                     -110.814056     0.158651  
                     -110.772385    -0.721063

In [45]:
# reset index

# loop through models
for model in models:
    maca_dfs[model]['date'] = maca_dfs[model].index.get_level_values('time')
    maca_dfs[model]['latitude'] = maca_dfs[model].index.get_level_values('lat')
    maca_dfs[model]['longitude'] = maca_dfs[model].index.get_level_values('lon')

# double check
maca_dfs['GFDL-ESM2M'].head()

ppt       tmax      tmin    tmean_c  \
time       lat       lon                                                
2045-10-01 44.437767 -112.897354  0.0  21.276672  0.786499  11.031586   
                     -110.897385  0.0  18.957458 -1.710175   8.623642   
                     -110.855713  0.0  20.251587 -2.481018   8.885284   
                     -110.814056  0.0  20.666504 -2.614594   9.025955   
                     -110.772385  0.0  19.401642 -2.049164   8.676239   

                                  water_year  dowy  wy_cumul_ppt_mm  \
time       lat       lon                                              
2045-10-01 44.437767 -112.897354        2046     1              0.0   
                     -110.897385        2046     1              0.0   
                     -110.855713        2046     1              0.0   
                     -110.814056        2046     1              0.0   
                     -110.772385        2046     1              0.0   

                                  tmin_roll3_c  tmax_roll3_c  ppt_roll3_mm  \
time       lat       lon                                                     
2045-10-01 44.437767 -112.897354      0.786499     21.276672           0.0   
                     -110.897385     -1.710175     18.957458           0.0   
                     -110.855713     -2.481018     20.251587           0.0   
                     -110.814056     -2.614594     20.666504           0.0   
                     -110.772385     -2.049164     19.401642           0.0   

                                  ...  ppt_roll7_mm  wy_cumul_pos_degree_days  \
time       lat       lon          ...                                           
2045-10-01 44.437767 -112.897354  ...           0.0                 11.031586   
                     -110.897385  ...           0.0                  8.623642   
                     -110.855713  ...           0.0                  8.885284   
                     -110.814056  ...           0.0                  9.025955   
                     -110.772385  ...           0.0                  8.676239   

                                    elevation      slope      aspect  \
time       lat       lon                                               
2045-10-01 44.437767 -112.897354  2612.297363  15.732635   87.257339   
                     -110.897385  2529.273438   3.932833  187.365433   
                     -110.855713  2438.088379   5.410627  255.512466   
                     -110.814056  2392.905762   5.706942  160.218735   
                     -110.772385  2478.738525   6.219458  179.198456   

                                  aspect_north  aspect_east       date  \
time       lat       lon                                                 
2045-10-01 44.437767 -112.897354     -0.065043     0.795005 2045-10-01   
                     -110.897385      0.105646     0.419466 2045-10-01   
                     -110.855713     -0.306499     0.789064 2045-10-01   
                     -110.814056      0.269574     0.158651 2045-10-01   
                     -110.772385      0.019351    -0.721063 2045-10-01   

                                   latitude   longitude  
time       lat       lon                                 
2045-10-01 44.437767 -112.897354  44.437767 -112.897354  
                     -110.897385  44.437767 -110.897385  
                     -110.855713  44.437767 -110.855713  
                     -110.814056  44.437767 -110.814056  
                     -110.772385  44.437767 -110.772385  

[5 rows x 22 columns]

In [47]:
# get final arrangement of dfs before saving

# model features order
model_features_in_order = [
    # date
    'date',
    # lat and lon
    'latitude', 'longitude', 
    # raw prism data
    'ppt_mm', 'tmin_c', 'tmax_c', 
    # topographic data
    'elevation', 'slope', 'aspect_north', 'aspect_east', 
    # lagged prism variables 7-day
    'tmin_c_roll_7d', 'tmax_c_roll_7d', 'ppt_mm_roll_7d',
    # cumulative precip and positive degree days
    'wy_cumul_precip_mm', 'wy_cumul_melt_days',
    # day of water year
    'dowy']

# initialize a dicts
maca_rnm_dfs = {}
maca_reord_dfs = {}

# loop through dfs, renaming cols and reordering
for model in models:
    # rename cols
    maca_reord_dfs[model] = maca_dfs[model].rename(columns = {
        'ppt': 'ppt_mm', 
        'tmin': 'tmin_c',
        'tmax': 'tmax_c',
        'tmin_roll7_c': 'tmin_c_roll_7d',
        'tmax_roll7_c': 'tmax_c_roll_7d',
        'ppt_roll7_mm': 'ppt_mm_roll_7d',
        'wy_cumul_pos_degree_days': 'wy_cumul_melt_days',
        'wy_cumul_ppt_mm': 'wy_cumul_precip_mm'})
        #inplace = True)

    # reorder the dfs to match the ML data
    maca_reord_dfs[model] = maca_reord_dfs[model][model_features_in_order]

In [56]:
# check order
maca_reord_dfs['GFDL-ESM2M'].head()

date   latitude   longitude  ppt_mm  \
time       lat       lon                                                     
2045-10-01 44.437767 -112.897354 2045-10-01  44.437767 -112.897354     0.0   
                     -110.897385 2045-10-01  44.437767 -110.897385     0.0   
                     -110.855713 2045-10-01  44.437767 -110.855713     0.0   
                     -110.814056 2045-10-01  44.437767 -110.814056     0.0   
                     -110.772385 2045-10-01  44.437767 -110.772385     0.0   

                                    tmin_c     tmax_c    elevation      slope  \
time       lat       lon                                                        
2045-10-01 44.437767 -112.897354  0.786499  21.276672  2612.297363  15.732635   
                     -110.897385 -1.710175  18.957458  2529.273438   3.932833   
                     -110.855713 -2.481018  20.251587  2438.088379   5.410627   
                     -110.814056 -2.614594  20.666504  2392.905762   5.706942   
                     -110.772385 -2.049164  19.401642  2478.738525   6.219458   

                                  aspect_north  aspect_east  tmin_c_roll_7d  \
time       lat       lon                                                      
2045-10-01 44.437767 -112.897354     -0.065043     0.795005        0.786499   
                     -110.897385      0.105646     0.419466       -1.710175   
                     -110.855713     -0.306499     0.789064       -2.481018   
                     -110.814056      0.269574     0.158651       -2.614594   
                     -110.772385      0.019351    -0.721063       -2.049164   

                                  tmax_c_roll_7d  ppt_mm_roll_7d  \
time       lat       lon                                           
2045-10-01 44.437767 -112.897354       21.276672             0.0   
                     -110.897385       18.957458             0.0   
                     -110.855713       20.251587             0.0   
                     -110.814056       20.666504             0.0   
                     -110.772385       19.401642             0.0   

                                  wy_cumul_precip_mm  wy_cumul_melt_days  dowy  
time       lat       lon                                                        
2045-10-01 44.437767 -112.897354                 0.0           11.031586     1  
                     -110.897385                 0.0            8.623642     1  
                     -110.855713                 0.0            8.885284     1  
                     -110.814056                 0.0            9.025955     1  
                     -110.772385                 0.0            8.676239     1

In [ ]:
# check for NaNs
for model in models:
    print(f'{model}: NaNs per column:')
    print(maca_reord_dfs[model].isna().sum())

# it's obviously a little late to check this, but I assumed that MACA data wouldn't have NaNs - that's confirmed here.

CNRM-CM5: NaNs per column:
date                  0
latitude              0
longitude             0
ppt_mm                0
tmin_c                0
tmax_c                0
elevation             0
slope                 0
aspect_north          0
aspect_east           0
tmin_c_roll_7d        0
tmax_c_roll_7d        0
ppt_mm_roll_7d        0
wy_cumul_precip_mm    0
wy_cumul_melt_days    0
dowy                  0
dtype: int64
GFDL-ESM2M: NaNs per column:
date                  0
latitude              0
longitude             0
ppt_mm                0
tmin_c                0
tmax_c                0
elevation             0
slope                 0
aspect_north          0
aspect_east           0
tmin_c_roll_7d        0
tmax_c_roll_7d        0
ppt_mm_roll_7d        0
wy_cumul_precip_mm    0
wy_cumul_melt_days    0
dowy                  0
dtype: int64
CanESM2: NaNs per column:
date                  0
latitude              0
longitude             0
ppt_mm                0
tmin_c                0
tmax

## Step 4: Save cleaned and engineered MACA data for ML usage

In [53]:
# set MACA dir for saving dfs
ml_save_dir = Path(cleaned_data_dir, 'machine_learning')
maca_ml_save_dir = Path(ml_save_dir, 'maca')
os.makedirs(maca_ml_save_dir, exist_ok=True)

In [54]:
# save MACA dfs to maca ml dir

# loop through each
for model in models:
    print(f'Saving {model}.')
    # set new path
    save_path = Path(maca_ml_save_dir, f'{model}_ML_features.parquet')
    # save to parquet
    maca_reord_dfs[model].to_parquet(save_path)
    print(f'{model} saved.')

Saving CNRM-CM5
CNRM-CM5 saved.
Saving GFDL-ESM2M
GFDL-ESM2M saved.
Saving CanESM2
CanESM2 saved.
Saving IPSL-CM5A-MR
IPSL-CM5A-MR saved.
